Ticket Booking Agent

Core Functionalities
- Book Tickets
- Cancle Tickets
- Get Ticket Details
- Refund Payment for the cancelled ticket

In this all the tools are binded there with agent but HITL is noot implemented, HITL is implemented in the extension of v0.

In [64]:
from dotenv import load_dotenv
from pathlib import Path
import os

env_path = Path.cwd().parent.parent / ".env"

api_key = os.getenv("OPENAI_API_KEY")


In [65]:
load_dotenv(env_path)
api_key = os.getenv("OPENAI_API_KEY")

In [66]:
from dataclasses import dataclass
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from langchain.tools import tool, ToolRuntime
from langgraph.store.memory import InMemoryStore
from langgraph.checkpoint.memory import InMemorySaver
from pydantic import BaseModel, Field
import uuid

In [67]:
store = InMemoryStore()
checkpointer = InMemorySaver()

In [68]:
llm = ChatOpenAI(
    model="gpt-4.1-mini",
    temperature=0,
    api_key=api_key
)

In [69]:
# Do not book the 0 tickets or more than 10 tickets

In [70]:
class TicketRequest(BaseModel):
    tickets: int = Field(
        description="Number of movie tickets to book"
    )
    movie_name: str = Field(
        description="Name of the movie"
    )


@tool
def book_ticket(
    request_details: TicketRequest,
    runtime: ToolRuntime
):
    """
    Book movie tickets for the user.
    """

    user_name = runtime.context.user_name

    # Read existing bookings
    item = runtime.store.get(
        "user-ticket-map",
        user_name
    )

    tickets = item.value if item else []

    # Pydantic has already validated these
    number_of_tickets = request_details.tickets
    movie_name = request_details.movie_name

    if number_of_tickets <= 0 or number_of_tickets > 10:
        return "Please book between 1 and 10 tickets only."

    # Generate ticket IDs
    ticket_ids = [
        str(uuid.uuid4())
        for _ in range(number_of_tickets)
    ]

    ticket_obj = {
        "total_tickets": number_of_tickets,
        "ticket_ids": ticket_ids,
        "movie_name": movie_name,
    }

    tickets.append(ticket_obj)

    # Persist
    runtime.store.put(
        "user-ticket-map",
        user_name,
        tickets
    )

    return (
        f"Hello {user_name}, {number_of_tickets} tickets "
        f"have been booked for {movie_name}. "
        f"Ticket IDs: {ticket_ids}"
    )

In [71]:
@tool
def get_ticket_details(runtime: ToolRuntime):
    """
    getting the list of booked tickets by the user
    """

    user_name = runtime.context.user_name
    ticket_item = runtime.store.get("user-ticket-map", user_name)
    ticket_list = ticket_item.value if ticket_item else []

    if len(ticket_list) == 0:
        return "no booking has been made yet"
    else:
        return ticket_list


In [80]:
@tool
def cancel_booking(ticket_id: str, runtime: ToolRuntime):
    """
    Cancel a movie ticket using its ticket ID.
    The ticket can only be cancelled by the user who owns it.
    """

    user_name = runtime.context.user_name

    # 1. Get user's existing bookings
    item = runtime.store.get(
        "user-ticket-map",
        user_name
    )

    if item is None:
        return f"No bookings found for {user_name}."

    ticket_list = item.value

    if not ticket_list:
        return f"No bookings found for {user_name}."

    # 2. Search for the ticket ID
    for booking in ticket_list:

        if ticket_id in booking["ticket_ids"]:

            # Save information for response
            movie_name = booking["movie_name"]

            # 3. Remove the ticket ID
            booking["ticket_ids"].remove(ticket_id)

            # 4. Decrease ticket count
            booking["total_tickets"] -= 1

            # 5. If no tickets remain, remove the booking
            if booking["total_tickets"] == 0:
                ticket_list.remove(booking)

            # 6. Persist updated bookings
            runtime.store.put(
                "user-ticket-map",
                user_name,
                ticket_list
            )

            return (
                f"Ticket {ticket_id} has been cancelled successfully "
                f"for movie '{movie_name}'."
            )

    # 7. Ticket wasn't found
    return (
        f"Ticket ID '{ticket_id}' was not found in your bookings."
    )

In [72]:
@dataclass
class Context:
    user_name:str
    email:str

In [82]:
agent = create_agent(
    model=llm,
    tools=[book_ticket, get_ticket_details, cancel_booking],
    store=store,
    context_schema=Context,
    checkpointer=checkpointer
)

In [74]:
config = {
    "configurable": {
        "thread_id": "user-uttam"
    }
}

In [77]:
result = agent.invoke(
    {
        "messages": [
            {"role": "user", "content": "Book 2 tickets for the movie Ramayan"}
        ]
    },
    config=config,
    context=Context(
        user_name="uttam kumar",
        email="uttamkumar@gmail.com"
    )
)

print(result)
print(result["messages"][-1].content)

d:\SoftwareEngineering\ai_ml\generativeAndAgeneticAI\Langchain-Langgraph\venv\Lib\site-packages\pydantic\functional_validators.py:835: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='context', input_value=Context(user_name='uttam ...='uttamkumar@gmail.com'), input_type=Context])
  function=lambda v, h: h(v), schema=original_schema
d:\SoftwareEngineering\ai_ml\generativeAndAgeneticAI\Langchain-Langgraph\venv\Lib\site-packages\pydantic\main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='context', input_value=Context(user_name='uttam ...='uttamkumar@gmail.com'), input_type=Context])
  return self.__pydantic_serializer__.to_python(


{'messages': [HumanMessage(content='Book 2 tickets for the movie Ramayan', additional_kwargs={}, response_metadata={}, id='77d23a62-1912-4867-9858-75a5f2b67fc5'), HumanMessage(content='Book 2 tickets for the movie Ramayan', additional_kwargs={}, response_metadata={}, id='cd757f4c-e014-43d7-8422-606bafd3fb8e'), HumanMessage(content='Book 2 tickets for the movie Ramayan', additional_kwargs={}, response_metadata={}, id='31a48f6e-6948-445a-a191-fb077b36dd22'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 23, 'prompt_tokens': 121, 'total_tokens': 144, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-mini-2025-04-14', 'system_fi

In [79]:
result_v1 = agent.invoke(
    {
        "messages": [
            {"role": "user", "content": "Get the ticket details"}
        ]
    },
    config=config,
    context=Context(
        user_name="uttam kumar",
        email="uttamkumar@gmail.com"
    )
)

print(result_v1)
print(result_v1["messages"][-1].content)

d:\SoftwareEngineering\ai_ml\generativeAndAgeneticAI\Langchain-Langgraph\venv\Lib\site-packages\pydantic\functional_validators.py:835: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='context', input_value=Context(user_name='uttam ...='uttamkumar@gmail.com'), input_type=Context])
  function=lambda v, h: h(v), schema=original_schema
d:\SoftwareEngineering\ai_ml\generativeAndAgeneticAI\Langchain-Langgraph\venv\Lib\site-packages\pydantic\main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='context', input_value=Context(user_name='uttam ...='uttamkumar@gmail.com'), input_type=Context])
  return self.__pydantic_serializer__.to_python(


{'messages': [HumanMessage(content='Book 2 tickets for the movie Ramayan', additional_kwargs={}, response_metadata={}, id='77d23a62-1912-4867-9858-75a5f2b67fc5'), HumanMessage(content='Book 2 tickets for the movie Ramayan', additional_kwargs={}, response_metadata={}, id='cd757f4c-e014-43d7-8422-606bafd3fb8e'), HumanMessage(content='Book 2 tickets for the movie Ramayan', additional_kwargs={}, response_metadata={}, id='31a48f6e-6948-445a-a191-fb077b36dd22'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 23, 'prompt_tokens': 121, 'total_tokens': 144, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-mini-2025-04-14', 'system_fi

In [83]:
result_v1 = agent.invoke(
    {
        "messages": [
            {"role": "user", "content": "Cancel the tickets"}
        ]
    },
    config=config,
    context=Context(
        user_name="uttam kumar",
        email="uttamkumar@gmail.com"
    )
)

print(result_v1)
print(result_v1["messages"][-1].content)

d:\SoftwareEngineering\ai_ml\generativeAndAgeneticAI\Langchain-Langgraph\venv\Lib\site-packages\pydantic\functional_validators.py:835: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='context', input_value=Context(user_name='uttam ...='uttamkumar@gmail.com'), input_type=Context])
  function=lambda v, h: h(v), schema=original_schema
d:\SoftwareEngineering\ai_ml\generativeAndAgeneticAI\Langchain-Langgraph\venv\Lib\site-packages\pydantic\main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='context', input_value=Context(user_name='uttam ...='uttamkumar@gmail.com'), input_type=Context])
  return self.__pydantic_serializer__.to_python(


{'messages': [HumanMessage(content='Book 2 tickets for the movie Ramayan', additional_kwargs={}, response_metadata={}, id='77d23a62-1912-4867-9858-75a5f2b67fc5'), HumanMessage(content='Book 2 tickets for the movie Ramayan', additional_kwargs={}, response_metadata={}, id='cd757f4c-e014-43d7-8422-606bafd3fb8e'), HumanMessage(content='Book 2 tickets for the movie Ramayan', additional_kwargs={}, response_metadata={}, id='31a48f6e-6948-445a-a191-fb077b36dd22'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 23, 'prompt_tokens': 121, 'total_tokens': 144, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-mini-2025-04-14', 'system_fi